# THRESHOLD — TippingPointClassifier Training
**Run on NVIDIA Brev.dev** (GPU instance recommended: `g4dn.xlarge` or `a10g`)

This notebook:
1. Pulls 895K rows of real CalCOFI oceanographic data + GDELT geopolitical scores from Snowflake
2. Generates scientifically-grounded `threshold_proximity_score` labels from NOAA/WHO thresholds
3. Trains a `RandomForestRegressor` — GPU-accelerated via RAPIDS cuML if available, sklearn fallback
4. Saves `tipping_point_classifier.json` weights for the live backend to load

In [ ]:
# ── Install deps (run once) ──────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'snowflake-connector-python[pandas]', 'scikit-learn', 'numpy', 'pandas', 'tqdm'])

# Try RAPIDS cuML for GPU-accelerated RF (available on NVIDIA Brev instances)
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'cuml-cu12', '--extra-index-url', 'https://pypi.nvidia.com'], timeout=120)
    print('cuML installed')
except Exception:
    print('cuML not available — will use sklearn (CPU)')

In [ ]:
# ── Config — fill in your Snowflake creds ────────────────────────────────────
import os

SF_USER     = os.getenv('SNOWFLAKE_USER',     'PRS016')
SF_PASSWORD = os.getenv('SNOWFLAKE_PASSWORD', '')        # set as env var, never hardcode
SF_ACCOUNT  = os.getenv('SNOWFLAKE_ACCOUNT',  'TWBDGDJ-KWB81325')
SF_DATABASE = os.getenv('SNOWFLAKE_DATABASE', 'THRESHOLD_DB')
SF_SCHEMA   = os.getenv('SNOWFLAKE_SCHEMA',   'PUBLIC')
SF_WH       = os.getenv('SNOWFLAKE_WAREHOUSE','COMPUTE_WH')

OUTPUT_PATH = 'tipping_point_classifier.json'   # will copy back to ml/models/

assert SF_PASSWORD, 'Set SNOWFLAKE_PASSWORD env var before running'

In [ ]:
# ── Pull data from Snowflake ─────────────────────────────────────────────────
import snowflake.connector
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print('Connecting to Snowflake...')
conn = snowflake.connector.connect(
    user=SF_USER, password=SF_PASSWORD, account=SF_ACCOUNT,
    database=SF_DATABASE, schema=SF_SCHEMA, warehouse=SF_WH,
)
print('Connected.')

In [ ]:
# Pull CalCOFI features — all non-null rows with core oceanographic columns
print('Fetching CALCOFI_TSUNAMI_FEATURES...')
calcofi_sql = """
SELECT
    LAT_DEC, LON_DEC, DATE,
    T_DEGC, O2ML_L, SALNTY, CHLORA, NO3UM, PO4UM, SIO3UM, WIND_SPD, BAROMETER
FROM CALCOFI_TSUNAMI_FEATURES
WHERE T_DEGC IS NOT NULL
  AND O2ML_L IS NOT NULL
  AND SALNTY IS NOT NULL
ORDER BY RANDOM()
LIMIT 200000
"""
calcofi_df = pd.read_sql(calcofi_sql, conn)
calcofi_df.columns = [c.lower() for c in calcofi_df.columns]
print(f'CalCOFI rows: {len(calcofi_df):,}')

In [ ]:
# Pull GDELT geopolitical scores — aggregate per 2-degree cell to join by location
print('Fetching GDELT aggregates...')
gdelt_sql = """
SELECT
    ROUND(ACTIONGEOLAT / 2) * 2  AS lat_cell,
    ROUND(ACTIONGEOLONG / 2) * 2 AS lon_cell,
    AVG(GOLDSTEIN)               AS goldstein,
    AVG(NUMARTS)                 AS numarts
FROM GDELT
WHERE ACTIONGEOLAT IS NOT NULL
  AND ACTIONGEOLONG IS NOT NULL
GROUP BY 1, 2
"""
gdelt_df = pd.read_sql(gdelt_sql, conn)
gdelt_df.columns = [c.lower() for c in gdelt_df.columns]
print(f'GDELT grid cells: {len(gdelt_df):,}')
conn.close()

In [ ]:
# ── Join CalCOFI with GDELT by 2-degree grid cell ───────────────────────────
calcofi_df['lat_cell'] = (calcofi_df['lat_dec'] / 2).round() * 2
calcofi_df['lon_cell'] = (calcofi_df['lon_dec'] / 2).round() * 2

df = calcofi_df.merge(gdelt_df, on=['lat_cell', 'lon_cell'], how='left')
df['goldstein'] = df['goldstein'].fillna(0.0)
df['numarts']   = df['numarts'].fillna(0.0)

# Fill remaining feature NaNs with oceanographic median defaults
DEFAULTS = {'chlora': 0.5, 'no3um': 5.0, 'po4um': 0.8, 'sio3um': 10.0,
            'wind_spd': 10.0, 'barometer': 1013.0}
for col, val in DEFAULTS.items():
    df[col] = df[col].fillna(val)

print(f'Joined dataset: {len(df):,} rows')
df.head(3)

In [ ]:
# ── Generate domain-derived labels (Option 3) ────────────────────────────────
#
# Each sub-score is grounded in published thresholds:
#
#  thermal_stress (0-3.0)
#    NOAA bleaching alert: MMM threshold ~29°C for tropics, ~16°C temperate.
#    We use 25°C as a generalised stress onset, full stress at 32°C.
#
#  hypoxia_stress (0-2.5)
#    WHO/NOAA: <2.0 ml/L = hypoxic dead zone; <1.4 = severe.
#    Healthy baseline is ~6–8 ml/L.
#
#  eutrophication (0-1.5)
#    Chlorophyll >10 mg/m³ indicates harmful algal bloom risk (EPA).
#
#  nutrient_loading (0-1.0)
#    Elevated nitrates (>30 µmol/L) drive dead zones (NOAA NCCOS).
#
#  salinity_stress (0-1.0)
#    Deviation from global mean (~33.5 PSU): freshwater influx or
#    hypersaline stress both degrade ecosystems.
#
#  geopolitical_instability (0-1.0)
#    Negative Goldstein score → conflict/instability compounds ecological stress.
#    GDELT Goldstein range: -10 (most destabilising) to +10 (most cooperative).

def label_score(row):
    # Thermal stress — NOAA coral bleaching thresholds
    thermal   = np.clip((row['t_degc'] - 25.0) / 7.0, 0, 1) * 3.0

    # Hypoxia — inverted O2 (low O2 = high stress)
    hypoxia   = np.clip((6.0 - row['o2ml_l']) / 6.0, 0, 1) * 2.5

    # Eutrophication via chlorophyll
    eutroph   = np.clip(row['chlora'] / 10.0, 0, 1) * 1.5

    # Nutrient loading via nitrate
    nutrient  = np.clip(row['no3um'] / 30.0, 0, 1) * 1.0

    # Salinity deviation from 33.5 PSU
    salinity  = np.clip(abs(row['salnty'] - 33.5) / 5.0, 0, 1) * 1.0

    # Geopolitical instability (Goldstein: -10 worst → +10 best)
    geopol    = np.clip((-row['goldstein']) / 10.0, 0, 1) * 1.0

    return round(thermal + hypoxia + eutroph + nutrient + salinity + geopol, 4)

print('Computing labels...')
df['threshold_proximity_score'] = df.apply(label_score, axis=1)
df['threshold_proximity_score'] = df['threshold_proximity_score'].clip(0, 10)

print(df['threshold_proximity_score'].describe())
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
df['threshold_proximity_score'].hist(bins=40, figsize=(10,4))
plt.title('Label distribution')
plt.xlabel('threshold_proximity_score')
plt.savefig('label_distribution.png', dpi=80)
plt.show()
print('Label distribution saved to label_distribution.png')

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
FEATURE_COLUMNS = [
    't_degc', 'o2ml_l', 'salnty', 'chlora', 'no3um',
    'po4um', 'sio3um', 'wind_spd', 'barometer', 'goldstein', 'numarts'
]

X = df[FEATURE_COLUMNS].fillna(0.0).values.astype('float32')
y = df['threshold_proximity_score'].values.astype('float32')

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

# Try GPU (cuML), fall back to sklearn
GPU_AVAILABLE = False
try:
    import cuml
    from cuml.ensemble import RandomForestRegressor
    import cudf
    X_train_g = cudf.DataFrame(X_train, columns=FEATURE_COLUMNS)
    y_train_g = cudf.Series(y_train)
    GPU_AVAILABLE = True
    print('Using cuML GPU RandomForest')
except Exception as e:
    from sklearn.ensemble import RandomForestRegressor
    print(f'Using sklearn CPU RandomForest (cuML: {e})')

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_streams=4 if GPU_AVAILABLE else 1,
    n_jobs=-1 if not GPU_AVAILABLE else None,
)

print('Training...')
if GPU_AVAILABLE:
    rf.fit(X_train_g, y_train_g)
else:
    rf.fit(X_train, y_train)
print('Training complete.')

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

if GPU_AVAILABLE:
    X_test_g = cudf.DataFrame(X_test, columns=FEATURE_COLUMNS)
    preds = rf.predict(X_test_g).to_numpy()
else:
    preds = rf.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

# Feature importances
if GPU_AVAILABLE:
    importances = rf.feature_importances_.to_numpy()
else:
    importances = rf.feature_importances_

for feat, imp in sorted(zip(FEATURE_COLUMNS, importances), key=lambda x: -x[1]):
    print(f'  {feat:<12} {imp:.4f}')

In [ ]:
# ── Save JSON weights (backend-compatible format) ────────────────────────────
import json
import pandas as pd

X_df = pd.DataFrame(X_train, columns=FEATURE_COLUMNS)

payload = {
    'feature_columns': FEATURE_COLUMNS,
    'feature_baselines': {col: float(X_df[col].median()) for col in FEATURE_COLUMNS},
    'feature_spans': {
        col: float(max(X_df[col].max() - X_df[col].min(), 1.0))
        for col in FEATURE_COLUMNS
    },
    'feature_importances': {
        col: float(imp) for col, imp in zip(FEATURE_COLUMNS, importances)
    },
    'metrics': {'rmse': round(rmse, 4), 'mae': round(mae, 4), 'r2': round(r2, 4)},
    'is_trained': True,
    'training_rows': int(len(X_train)),
    'gpu_trained': GPU_AVAILABLE,
}

with open(OUTPUT_PATH, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved to {OUTPUT_PATH}')
print(json.dumps(payload, indent=2))

In [ ]:
# ── Download the weights file ─────────────────────────────────────────────────
# In Brev/JupyterLab: right-click the file in the file browser and Download.
# Then place it at:  threshold/ml/models/tipping_point_classifier.json
#
# Or if you have the repo cloned on Brev, copy directly:
#   cp tipping_point_classifier.json /path/to/threshold/ml/models/

print('Done! Download tipping_point_classifier.json and place it in ml/models/')
print(f'R² = {r2:.4f}  |  RMSE = {rmse:.4f}  |  Rows trained: {len(X_train):,}')